<h3 style="color: #777777;"> Paso 0: Validar la ubicación de la base de datos</h3>

In [3]:
# MINIMAL DEBUG SCRIPT
import sqlite3
from pathlib import Path
import pandas as pd

print("1. Current directory:", Path.cwd())
print("\n2. Looking for nursery.db...")

# Try exact paths that should work
paths_to_try = [
    '../nursery.db',  # Most likely if you're in 'reports' folder
    'nursery.db',
    '../../nursery.db',
]

for path_str in paths_to_try:
    path = Path(path_str)
    print(f"\nTrying: {path_str}")
    print(f"  Absolute: {path.absolute()}")
    print(f"  Exists: {path.exists()}")
    
    if path.exists():
        try:
            conn = sqlite3.connect(str(path))
            cursor = conn.cursor()
            
            # List ALL tables
            cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
            tables = [t[0] for t in cursor.fetchall()]
            
            print(f"  ✅ Connected! Tables: {tables}")
            
            # If OBSERVACION_LOTE exists, show sample
            if 'OBSERVACION_LOTE' in tables:
                cursor.execute("SELECT * FROM OBSERVACION_LOTE LIMIT 3")
                sample = cursor.fetchall()
                print(f"  Sample from OBSERVACION_LOTE: {sample}")
            else:
                # Check for any observation-like table
                for table in tables:
                    if 'OBSERV' in table.upper():
                        cursor.execute(f"SELECT * FROM {table} LIMIT 3")
                        sample = cursor.fetchall()
                        print(f"  Sample from {table}: {sample}")
            
            conn.close()
            break
            
        except Exception as e:
            print(f"  ❌ Error: {e}")

1. Current directory: /Users/estefania/Documents/Data/my_portfolio_projects/scripts/nursery_manager

2. Looking for nursery.db...

Trying: ../nursery.db
  Absolute: /Users/estefania/Documents/Data/my_portfolio_projects/scripts/nursery_manager/../nursery.db
  Exists: False

Trying: nursery.db
  Absolute: /Users/estefania/Documents/Data/my_portfolio_projects/scripts/nursery_manager/nursery.db
  Exists: True
  ✅ Connected! Tables: ['sqlite_sequence', 'GASTOS', 'REGISTRO_CLIMA', 'ENTRADA_PLANTAS', 'ETAPA_CRECIMIENTO', 'OBSERVACION_LOTE', 'REGISTRO_RIEGO', 'METODOS_PROPAGACION', 'CONTROL_PLAGAS', 'OBSERVACION_LOTE_BACKUP_12_19_25', 'ESPECIE_LOTE', 'CATEGORIA']
  Sample from OBSERVACION_LOTE: [(1, 6, '2025-09-22', 76, 20.0, 'BOLSA_PEQUEÑA', 'Buena', 'Requerimiento control de plagas y división de 6 papayos'), (2, 4, '2025-09-23', 39, 15.0, 'BOLSA_PEQUEÑA', 'Buena', 'Tiene algunas hojas deformes'), (3, 3, '2025-09-24', 88, 33.0, 'PRE-EMBOLSE', 'Buena', 'Algunas hojas con huecos y bordes café y

In [4]:
import sqlite3
import pandas as pd
from pathlib import Path
from IPython.display import HTML, display

def get_observation_data(start_date=None, end_date=None, db_path='nursery.db'):
    """
    Read observation data from the OBSERVACION_LOTE table with optional date filtering.
    """
    # Convert to Path object and resolve relative path
    db_file = Path(db_path)
    
    # Check if database exists
    if not db_file.exists():
        raise FileNotFoundError(f"Database file not found: {db_file.absolute()}")
    
    try:
        # Connect to database 
        conn = sqlite3.connect(str(db_file))

        # Build the query with optional date filtering
        query = """ 
        SELECT * FROM OBSERVACION_LOTE_BACKUP_12_19_25
        WHERE 1=1 -- Always true makes it easier to add conditions
        """

        # Add date filters if provided
        params = []
        if start_date:
            query += " AND FECHA_OBSERVACION >= ?"
            params.append(start_date)
        if end_date:
            query += " AND FECHA_OBSERVACION <= ?"
            params.append(end_date)

        # Order by date (most recent first)
        query += " ORDER BY FECHA_OBSERVACION DESC"

        # Execute the query and get data
        df = pd.read_sql_query(query, conn, params=params if params else None)
        
        return df
    
    except sqlite3.Error as e:
        print(f"Database error: {e}")
        raise
    finally:
        if 'conn' in locals():
            conn.close()

def generate_html_report(df, title="VIVERO RAÍCES DORADAS - REPORTE OPERATIVO"):
    """
    Generate an HTML report from observation data.
    
    Args:
        df (pandas.DataFrame): Observation data
        title (str): Report title
    
    Returns:
        str: HTML string
    """
    if df.empty:
        return "<h3>No data available</h3>"
    
    # Calculate statistics
    total_observations = len(df)
    date_range = f"{df['FECHA_OBSERVACION'].min()} a {df['FECHA_OBSERVACION'].max()}"
    unique_lots = df['ESPECIE_LOTE_ID'].nunique()
    available_columns = len(df.columns)
    
    # Create the HTML
    html_content = f'''
    <center><h1 style="color: #777777;">{title}</h1></center>
    <center><h2 style="color: #777777;">Observación de Plantas</h2></center>
    
    <div align="center">
        <table style="width: 80%; border-collapse: collapse; margin: 20px auto; font-family: Arial, sans-serif;">
            <tr>
                <th style="background-color: #e3d0b3; color: #926e41; padding: 12px; border: 1px solid #926e41;">Rango de Fechas</th>
                <th style="background-color: #e3d0b3; color: #926e41; padding: 12px; border: 1px solid #926e41;">Conteo de Observaciones</th>
                <th style="background-color: #e3d0b3; color: #926e41; padding: 12px; border: 1px solid #926e41;">Lotes Existentes</th>
                <th style="background-color: #e3d0b3; color: #926e41; padding: 12px; border: 1px solid #926e41;">Columnas disponibles</th>
            </tr>
            <tr>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #555;">{date_range}</td>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #555;">{total_observations:,}</td>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #555;">{unique_lots}</td>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #555;">{available_columns}</td>
            </tr>
        </table>
    </div>
    
    <div align="center" style="margin: 20px;">
        <h3 style="color: #777777;">Resumen histórico de datos</h3>
        <table style="width: 60%; border-collapse: collapse; margin: 10px auto; font-family: Arial, sans-serif;">
            <tr>
                <th style="background-color: #f5f5f5; color: #555; padding: 8px; border: 1px solid #ddd;">Nombre de Columna</th>
                <th style="background-color: #f5f5f5; color: #555; padding: 8px; border: 1px solid #ddd;">Valores Únicos</th>
                <th style="background-color: #f5f5f5; color: #555; padding: 8px; border: 1px solid #ddd;">Valores Totales</th>
            </tr>
    '''
    
    # Add row for each column
    for column in df.columns:
        unique_count = df[column].nunique()
        value_count = df[column].shape[0]
        
        html_content += f'''
            <tr>
                <td style="padding: 8px; border: 1px solid #ddd; color: #333;">{column}</td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: #666;">{unique_count}</td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: #666;">{value_count}</td>
            </tr>
        '''
    
    return html_content


def display_report(df=None, start_date=None, end_date=None, db_path='nursery.db'):
    """
    Complete function to get data and display HTML report.
    
    Args:
        df (pandas.DataFrame, optional): Pre-loaded data. If None, loads from database.
        start_date (str, optional): Start date for filtering
        end_date (str, optional): End date for filtering
        db_path (str, optional): Database path
    """
    from datetime import datetime
    
    # Get data if not provided
    if df is None:
        print("📊 Cargando datos de la base de datos...")
        df = get_observation_data(start_date=start_date, end_date=end_date, db_path=db_path)
        print(f"✅ Datos cargados: {len(df)} registros")
    
    # Generate and display HTML report
    html_report = generate_html_report(df)
    display(HTML(html_report) )
    
    # Return the dataframe for further analysis
    return df

# Create the current_inventory dataframe:
def get_current_inventory(df):
    """
    Extract current inventory from observation data.
    Returns the most recent observation for each ESPECIE_LOTE_ID and ETAPA_CRECIMINETO combination.
    """
    # Sort by date descending and keep first (most recent observation and stage per lot
    df_sorted = df.sort_values('FECHA_OBSERVACION', ascending=False)
    current_inventory = df_sorted.drop_duplicates(
        subset=['ESPECIE_LOTE_ID', 'ETAPA_CRECIMIENTO'], 
        keep='first'
    )
    print(f"✅ Inventario actual creado: {current_inventory.shape[0]} observaciones únicas")
    print(f"   Lotes únicos: {current_inventory['ESPECIE_LOTE_ID'].nunique()}")
    print(f"   Etapas de crecimiento únicas: {current_inventory['ETAPA_CRECIMIENTO'].nunique()}")
    
    return current_inventory


# Usage workflow
if __name__ == "__main__":
    #print("Iniciando reporte del vivero...\n")
    
    # Step 1: Display observation data
    print("Generando reporte filtrado...")
    observations = display_report(start_date='2025-10-27')

    # Step 2: Extract current inventory
    current_inventory = get_current_inventory(observations)
    

Generando reporte filtrado...
📊 Cargando datos de la base de datos...
✅ Datos cargados: 825 registros


✅ Inventario actual creado: 54 observaciones únicas
   Lotes únicos: 34
   Etapas de crecimiento únicas: 5
